## Task I: The classical Job Shop Problem

### 1. Read data

In [4]:
def read_instance(path):
    with open(path, "r") as f:
        lines = [line.strip() for line in f
                 if line.strip() and not line.strip().startswith("+")
                 and not line.strip().startswith("instance")
                 and not line.strip().startswith("Fisher")
                 and not line.strip().startswith("Lawrence")]

    n_jobs, n_machines = map(int, lines[0].split())

    jobs = []

    for line in lines[1:1 + n_jobs]:
        values = list(map(int, line.split()))

        operations = []
        for i in range(0, len(values), 2):
            machine = values[i]
            duration = values[i + 1]
            operations.append((machine, duration))

        jobs.append(operations)

    return n_jobs, n_machines, jobs

In [5]:
# print("Number of jobs:", n_jobs)
# print("Number of machines:", n_machines)
# print(jobs)

### 2.MILP Gurobi

In [6]:
import gurobipy as gp
from gurobipy import GRB
from itertools import combinations


def solve_milp(n_jobs, n_machines, jobs, time_limit=None):
    model = gp.Model("job_shop_milp")

    starts = {}
    for j in range(n_jobs):
        for h in range(len(jobs[j])):
            starts[j, h] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"start_{j}_{h}"
            )

    makespan = model.addVar(
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="makespan"
    )

    # all the operation in each machine
    machine_operations = {m: [] for m in range(n_machines)}

    for j in range(n_jobs):
        for h, (machine, duration) in enumerate(jobs[j]):
            machine_operations[machine].append((j, h, duration))

    #  Big-M：sum of all operation time
    big_m = sum(
        duration
        for job in jobs
        for _, duration in job
    )

    # job constraint
    for j in range(n_jobs):
        for h in range(len(jobs[j]) - 1):
            duration = jobs[j][h][1]

            model.addConstr(
                starts[j, h + 1] >= starts[j, h] + duration,
                name=f"precedence_{j}_{h}"
            )

    # machine mutex
    for machine, operations in machine_operations.items():
        for op_a, op_b in combinations(operations, 2):
            j1, h1, p1 = op_a
            j2, h2, p2 = op_b

            order = model.addVar(
                vtype=GRB.BINARY,
                name=f"order_{j1}_{h1}_{j2}_{h2}"
            )

            model.addConstr(
                starts[j1, h1] + p1
                <= starts[j2, h2] + big_m * (1 - order)
            )

            model.addConstr(
                starts[j2, h2] + p2
                <= starts[j1, h1] + big_m * order
            )

    # Makespan constraint
    for j in range(n_jobs):
        last_h = len(jobs[j]) - 1
        last_duration = jobs[j][last_h][1]

        model.addConstr(
            makespan >= starts[j, last_h] + last_duration,
            name=f"makespan_{j}"
        )

    model.setObjective(makespan, GRB.MINIMIZE)

    if time_limit is not None:
        model.setParam(GRB.Param.TimeLimit, time_limit)

    model.optimize()

    result = {
        "status": model.Status,
        "objective": None,
        "runtime": model.Runtime,
        "schedule": {}
    }

    if model.SolCount > 0:
        result["objective"] = makespan.X

        for j in range(n_jobs):
            for h in range(len(jobs[j])):
                result["schedule"][j, h] = starts[j, h].X

    return result

### 3. CP

In [7]:
import z3
from itertools import combinations


def solve_cp(n_jobs, n_machines, jobs, time_limit=None):
    solver = z3.Optimize()

    starts = {}
    for j in range(n_jobs):
        for h in range(len(jobs[j])):
            starts[j, h] = z3.Int(f"start_{j}_{h}")
            solver.add(starts[j, h] >= 0)

    makespan = z3.Int("makespan")
    solver.add(makespan >= 0)

    # operations within same job
    for j in range(n_jobs):
        for h in range(len(jobs[j]) - 1):
            duration = jobs[j][h][1]

            solver.add(
                starts[j, h + 1]
                >= starts[j, h] + duration
            )

    # operation according to machine
    machine_operations = {m: [] for m in range(n_machines)}

    for j in range(n_jobs):
        for h, (machine, duration) in enumerate(jobs[j]):
            machine_operations[machine].append((j, h, duration))

    # machine mutex
    for machine, operations in machine_operations.items():
        for op_a, op_b in combinations(operations, 2):
            j1, h1, p1 = op_a
            j2, h2, p2 = op_b

            solver.add(
                z3.Or(
                    starts[j1, h1] + p1 <= starts[j2, h2],
                    starts[j2, h2] + p2 <= starts[j1, h1]
                )
            )

    # Makespan
    for j in range(n_jobs):
        last_h = len(jobs[j]) - 1
        last_duration = jobs[j][last_h][1]

        solver.add(
            makespan >= starts[j, last_h] + last_duration
        )

    solver.minimize(makespan)

    if time_limit is not None:
        solver.set(timeout=time_limit * 1000)

    status = solver.check()

    result = {
        "status": str(status),
        "objective": None,
        "schedule": {}
    }

    if status in [z3.sat, z3.unknown]:
        model = solver.model()
        result["objective"] = model.evaluate(makespan).as_long()

        for j in range(n_jobs):
            for h in range(len(jobs[j])):
                result["schedule"][j, h] = (
                    model.evaluate(starts[j, h]).as_long()
                )

    return result

### 4. result

In [8]:
import csv
import time
import os

instance_names = ["ft06", "ft10", "la01", "la02", "la03", "la04", "la05"]

file_path = "task1_results.csv"
file_exists = os.path.exists(file_path)

# 已经跑过的 (instance, solver) 组合,避免重复执行、重复写入
done = set()
if file_exists:
    with open(file_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            done.add((row["instance"], row["solver"]))

with open(file_path, "a", newline="") as file:
    writer = csv.writer(file)

    if not file_exists:
        writer.writerow([
            "instance", "solver", "status", "makespan", "runtime_seconds"
        ])

    for instance_name in instance_names:
        instance_path = f"jobshop_data/{instance_name}"
        n_jobs, n_machines, jobs = read_instance(instance_path)

        # ---- Gurobi / MILP ----
        if (instance_name, "Gurobi") not in done:
            start_time = time.perf_counter()
            result_milp = solve_milp(n_jobs, n_machines, jobs, time_limit=1200)
            milp_runtime = time.perf_counter() - start_time

            writer.writerow([
                instance_name, "Gurobi",
                result_milp["status"], result_milp["objective"],
                round(milp_runtime, 4)
            ])
            file.flush()
            print(f"{instance_name} Gurobi: makespan={result_milp['objective']}, "
                  f"time={milp_runtime:.2f}s")

        # ---- Z3 / CP ----
        if (instance_name, "Z3") not in done:
            start_time = time.perf_counter()
            result_cp = solve_cp(n_jobs, n_machines, jobs, time_limit=1200)
            cp_runtime = time.perf_counter() - start_time

            writer.writerow([
                instance_name, "Z3",
                result_cp["status"], result_cp["objective"],
                round(cp_runtime, 4)
            ])
            file.flush()
            print(f"{instance_name} Z3: makespan={result_cp['objective']}, "
                  f"time={cp_runtime:.2f}s")

## Task II: The Extended Job Shop Problem

### define graph and nodes

In [9]:

graph = {
    "warehouse": {
        "j1": 7,
    },

    "j1": {
        "warehouse": 7,
        "j2": 17,
        "M5": 21,
        "M2": 97,
        "j13": 118
    },

    "j2": {
        "j3": 21,
        "M4": 40,
        "j1": 17
    },

    "j3": {
        "j2": 21,
        "j4": 25,
        "M5": 17
    },

    "M4": {
        "j2": 40,
        "j7": 45,
    },

    "M5": {
        "j1": 21,
        "j3": 17,
        "j5": 25,
    },

    "j4": {
        "j3": 25,
        "j5": 17,
        "M1": 20,
    },

    "j5": {
        "M5": 25,
        "j4": 17,
        "M2": 17,
    },

    "M2": {
        "j5": 17,
        "j6":37,
        "j1":97,
        "j13":55,
    },

    "j6": {
        "j10": 18,
        "M1": 17,
        "M2": 37,
    },

    "j7": {
        "M4": 45,
        "j8": 18,
        "M1": 19,
    },

    "j8": {
        "j9": 19,
        "j7": 18,
        "M6": 28,
    },

    "M6": {
        "j8": 28,
    },

    "M1": {
        "j4": 20,
        "j7": 19,
        "j9": 18,
        "j6": 17,
    },

    "M3": {
        "j10": 17,
        "j12": 28,
        "j13": 17,
    },

    "j9": {
        "M1": 18,
        "j10": 17,
        "j11": 45,
        "j8": 19,
    },

    "j10": {
        "j9": 17,
        "j6": 18,
        "j11": 28,
        "M3": 17,
    },

    "j11": {
        "j10": 28,
        "j9": 45,
        "delivery": 8,
        "j12": 17,
    },


     "j12": {
        "M3": 28,
        "j11": 17,
    },


     "j13": {
        "j1": 118,
        "M2": 55,
        "M3": 17,
        "j12": 45,
    },

    "delivery": {
        "j11": 8,
    }
}

In [10]:
coordinates = {
    "warehouse": (0, 34),
    "M1": (73, 51),
    "M2": (53, 17),
    "M3": (91, 17),
    "M4": (28, 70),
    "M5": (28, 34),
    "M6": (119, 70),
    "delivery": (127, 34),

    "j1": (7, 34),
    "j2": (7, 51),
    "j3": (28, 51),
    "j4": (53, 51),
    "j5": (53, 34),
    "j6": (73, 34),
    "j7": (73, 70),
    "j8": (91, 70),
    "j9": (91, 50),
    "j10": (91, 34),
    "j11": (119, 34),
    "j12": (119, 17),
    "j13": (91, 0),
}

### A* algorithm

In [11]:
import heapq
import math


def heuristic(node, goal, coordinates):
    x1, y1 = coordinates[node]
    x2, y2 = coordinates[goal]

    return math.sqrt(
        (x1 - x2) ** 2 +
        (y1 - y2) ** 2
    )


def reconstruct_path(parent, start, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    if path[0] != start:
        return None

    return path


def astar(graph, coordinates, start, goal):
    open_set = []

    # (f_score, counter, node)
    counter = 0
    heapq.heappush(
        open_set,
        (0, counter, start)
    )

    g_score = {
        node: float("inf")
        for node in graph
    }

    g_score[start] = 0

    parent = {
        start: None
    }

    closed_set = set()

    while open_set:
        _, _, current = heapq.heappop(open_set)

        if current in closed_set:
            continue

        if current == goal:
            path = reconstruct_path(
                parent,
                start,
                goal
            )

            return path, g_score[goal]

        closed_set.add(current)

        for neighbour, edge_cost in graph[current].items():
            tentative_g = (
                g_score[current] + edge_cost
            )

            if tentative_g < g_score[neighbour]:
                parent[neighbour] = current
                g_score[neighbour] = tentative_g

                f_score = (
                    tentative_g
                    + heuristic(
                        neighbour,
                        goal,
                        coordinates
                    )
                )

                counter += 1

                heapq.heappush(
                    open_set,
                    (f_score, counter, neighbour)
                )

    return None, float("inf")

### calculate the shortest path of all location

In [12]:
locations = [
    "warehouse",
    "M1",
    "M2",
    "M3",
    "M4",
    "M5",
    "M6",
    "delivery"
]

all_paths = {}
all_distances = {}
travel_times = {}

speed = 5.0  # metres per second

for start in locations:
    for goal in locations:
        if start == goal:
            all_paths[start, goal] = [start]
            all_distances[start, goal] = 0
            travel_times[start, goal] = 0
            continue

        path, distance = astar(
            graph,
            coordinates,
            start,
            goal
        )

        all_paths[start, goal] = path
        all_distances[start, goal] = distance
        travel_times[start, goal] = distance / speed

### print shortest path

In [13]:
for start in locations:
    for goal in locations:
        if start == goal:
            continue

        path = all_paths[start, goal]
        distance = all_distances[start, goal]
        travel_time = travel_times[start, goal]

        print(
            f"{start} -> {goal}: "
            f"path={path}, "
            f"distance={distance:.2f} m, "
            f"time={travel_time:.2f} s"
        )

warehouse -> M1: path=['warehouse', 'j1', 'M5', 'j5', 'j4', 'M1'], distance=90.00 m, time=18.00 s
warehouse -> M2: path=['warehouse', 'j1', 'M5', 'j5', 'M2'], distance=70.00 m, time=14.00 s
warehouse -> M3: path=['warehouse', 'j1', 'j13', 'M3'], distance=142.00 m, time=28.40 s
warehouse -> M4: path=['warehouse', 'j1', 'j2', 'M4'], distance=64.00 m, time=12.80 s
warehouse -> M5: path=['warehouse', 'j1', 'M5'], distance=28.00 m, time=5.60 s
warehouse -> M6: path=['warehouse', 'j1', 'M5', 'j5', 'j4', 'M1', 'j9', 'j8', 'M6'], distance=155.00 m, time=31.00 s
warehouse -> delivery: path=['warehouse', 'j1', 'M5', 'j5', 'j4', 'M1', 'j9', 'j11', 'delivery'], distance=161.00 m, time=32.20 s
M1 -> warehouse: path=['M1', 'j4', 'j3', 'j2', 'j1', 'warehouse'], distance=90.00 m, time=18.00 s
M1 -> M2: path=['M1', 'j6', 'M2'], distance=54.00 m, time=10.80 s
M1 -> M3: path=['M1', 'j6', 'j10', 'M3'], distance=52.00 m, time=10.40 s
M1 -> M4: path=['M1', 'j7', 'M4'], distance=64.00 m, time=12.80 s
M1 -> M

### apply MILP

In [14]:
def solve_extended_milp(
    n_jobs,
    n_machines,
    jobs,
    travel_times,
    time_limit=None
):
    model = gp.Model("extended_job_shop_milp")

    starts = {}

    for j in range(n_jobs):
        for h in range(len(jobs[j])):
            starts[j, h] = model.addVar(
                lb=0,
                vtype=GRB.CONTINUOUS,
                name=f"start_{j}_{h}"
            )

    makespan = model.addVar(
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="makespan"
    )

    machine_operations = {
        m: [] for m in range(n_machines)
    }

    for j in range(n_jobs):
        for h, (machine, duration) in enumerate(jobs[j]):
            machine_operations[machine].append(
                (j, h, duration)
            )

    big_m = sum(
        duration
        for job in jobs
        for _, duration in job
    )

    # Warehouse to first machine
    for j in range(n_jobs):
        first_machine = jobs[j][0][0]

        model.addConstr(
            starts[j, 0]
            >= travel_times[
                "warehouse",
                f"M{first_machine+1}"
            ]
        )

    # Job sequence + transportation time
    for j in range(n_jobs):
        for h in range(len(jobs[j]) - 1):
            machine_current = jobs[j][h][0]
            duration_current = jobs[j][h][1]
            machine_next = jobs[j][h + 1][0]

            machine_current_name = f"M{machine_current + 1}"
            machine_next_name = f"M{machine_next + 1}"

            transport = travel_times[
            machine_current_name,
            machine_next_name
            ]



            model.addConstr(
                starts[j, h + 1]
                >= starts[j, h]
                + duration_current
                + transport
            )

    # mahchine mutex constraint
    for machine, operations in machine_operations.items():
        for op_a, op_b in combinations(operations, 2):
            j1, h1, p1 = op_a
            j2, h2, p2 = op_b

            order = model.addVar(
                vtype=GRB.BINARY,
                name=f"order_{j1}_{h1}_{j2}_{h2}"
            )

            model.addConstr(
                starts[j1, h1] + p1
                <= starts[j2, h2]
                + big_m * (1 - order)
            )

            model.addConstr(
                starts[j2, h2] + p2
                <= starts[j1, h1]
                + big_m * order
            )

    # last machine to Delivery
    for j in range(n_jobs):
        last_h = len(jobs[j]) - 1
        last_machine = jobs[j][last_h][0]
        last_duration = jobs[j][last_h][1]

        transport_to_delivery = travel_times[
            f"M{last_machine+1}",
            "delivery"
        ]

        model.addConstr(
            makespan
            >= starts[j, last_h]
            + last_duration
            + transport_to_delivery
        )

    model.setObjective(
        makespan,
        GRB.MINIMIZE
    )

    if time_limit is not None:
        model.setParam(
            GRB.Param.TimeLimit,
            time_limit
        )

    model.optimize()

    return model, starts, makespan

In [15]:
import time

n_jobs, n_machines, jobs = read_instance(
    "jobshop_data/ft06"
)

start_time = time.perf_counter()

model, starts, makespan = solve_extended_milp(
    n_jobs,
    n_machines,
    jobs,
    travel_times,
    time_limit=1200
)

runtime = time.perf_counter() - start_time

if model.SolCount > 0:
    print("Status:", model.Status)
    print("Extended makespan:", makespan.X)
    print("Runtime:", runtime)
else:
    print("No feasible solution found.")

Restricted license - for non-production use only - expires 2027-11-29
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1200

Optimize a model with 222 rows, 127 columns and 618 nonzeros (Min)
Model fingerprint: 0xdfac936e
Model has 1 linear objective coefficients
Variable types: 37 continuous, 90 integer (90 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+02]

Presolve removed 6 rows and 0 columns
Presolve time: 0.01s
Presolved: 216 rows, 127 columns, 612 nonzeros
Variable types: 37 continuous, 90 integer (90 binary)

Root relaxation: objective 1.764000e+02, 44 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |   

### save data to csv file

In [16]:
import csv

with open(
    "task2_shortest_paths.csv",
    "w",
    newline=""
) as file:
    writer = csv.writer(file)

    writer.writerow([
        "from",
        "to",
        "path",
        "distance_m",
        "travel_time_s"
    ])

    for start in locations:
        for goal in locations:
            if start == goal:
                continue

            writer.writerow([
                start,
                goal,
                " -> ".join(all_paths[start, goal]),
                round(all_distances[start, goal], 4),
                round(travel_times[start, goal], 4)
            ])

## Task II: The Extended Job Shop Problem

In [17]:
import heapq
import itertools
from z3 import Bool, Solver, Or, And, Not, Implies, sat, is_true
# define graph
nodevertical=5
nodehorizontal=8
costvertical=[19,17,17,17]
costhorizontal=[7,21,25,20,18,28,8]
# generate a grid 5 x 8 then remove some line
remove={"(0,0)-(0,1)", "(0,6)-(0,7)", "(1,0)-(1,1)", "(1,6)-(1,7)", "(2,3)-(2,4)",
    "(3,0)-(3,1)", "(3,2)-(3,3)", "(3,4)-(3,5)", "(3,6)-(3,7)", "(4,0)-(4,1)",
    "(4,6)-(4,7)", "(0,0)-(1,0)", "(0,2)-(1,2)", "(0,3)-(1,3)", "(0,6)-(1,6)",
    "(0,7)-(1,7)", "(1,0)-(2,0)", "(1,7)-(2,7)", "(2,0)-(3,0)", "(2,2)-(3,2)",
    "(2,7)-(3,7)", "(3,0)-(4,0)", "(3,4)-(4,4)", "(3,7)-(4,7)",}
# the position of the machine will be represented by the node
machine={"machine 1":"(1,4)","machine 2":"(3,3)","machine 3":"(3,5)",
         "machine 4":"(0,2)","machine 5":"(2,2)","machine 6":"(0,6)",
         "warehouse":"(0,0)", "Delivery": "(0,7)"}

edges=[]
weights={}
for i in range(nodevertical):
    for j in range(nodehorizontal-1):
        e=f"({i},{j})-({i},{j+1})"
        edges.append(e); weights[e]=costhorizontal[j]
for i in range(nodevertical-1):
    for j in range(nodehorizontal):
        e = f"({i},{j})-({i+1},{j})"
        edges.append(e); weights[e]=costvertical[i]
edges=[e for e in edges if e not in remove]
ends = {e: tuple(e.split("-")) for e in edges}
inc={}
for e, (u,v) in ends.items():
    inc.setdefault(u, []).append(e)
    inc.setdefault(v, []).append(e)

# Boolean model
def build_model(s, t, must_visit=(), must_avoid=()):
    E = {e: Bool(e) for e in edges}
    N = {v: Bool("n" + v) for v in inc}
    sol = Solver()

    for e, (u, v) in ends.items():   # chosen edge => both endpoints used
        sol.add(Implies(E[e], N[u]), Implies(E[e], N[v]))

    for v, es in inc.items():
        lits = [E[e] for e in es]
        if v in (s, t):       # exactly one incident edge
            sol.add(N[v], Or(lits))
            for a, b in itertools.combinations(lits, 2):
                sol.add(Not(And(a, b)))
        else:             # 0 or exactly 2 incident edges
            for a, b, c in itertools.combinations(lits, 3):
                sol.add(Not(And(a, b, c)))    # at most 2
            for k in range(len(lits)):   # N_v => at least 2
                sol.add(Implies(N[v], Or(lits[:k] + lits[k + 1:])))
    for v in must_visit:
        sol.add(N[v])
    for v in must_avoid:
        sol.add(Not(N[v]))
    return sol, E

def split_solution(used, s, t):

    adj = {}
    for e in used:
        u, v = ends[e]
        adj.setdefault(u, []).append((v, e))
        adj.setdefault(v, []).append((u, e))
    path, cur, prev = [], s, None
    while cur != t:
        nxt = [(v, e) for v, e in adj[cur] if e != prev]
        v, e = nxt[0]
        path.append(e); prev, cur = e, v
    rest = set(used) - set(path)
    cycles = []
    while rest:                                # connected components of the rest
        comp, stack = set(), [next(iter(rest))]
        while stack:
            e = stack.pop()
            if e in comp:
                continue
            comp.add(e)
            for n in ends[e]:
                stack += [x for x in inc[n] if x in rest and x not in comp]
        cycles.append(comp); rest -= comp
    return path, cycles


def enumerate_paths(s, t, **kw):
    sol, E = build_model(s, t, **kw)
    paths, n_cycle_hits = [], 0
    while sol.check() == sat:
        m = sol.model()
        used = [e for e in edges if is_true(m.eval(E[e], model_completion=True))]
        path, cycles = split_solution(used, s, t)
        for c in cycles:                       # forbid every detached cycle
            sol.add(Not(And([E[e] for e in c])))
      #  n_cycle_hits += bool(cycles)
        else:
            paths.append(path)
            sol.add(Not(And([E[e] for e in path])))  # block this path
    return paths, n_cycle_hits

def dijkstra(s, t):
    dist, pq = {s: 0}, [(0, s)]
    while pq:
        d, u = heapq.heappop(pq)
        if u == t:
            return d
        if d > dist[u]:
            continue
        for e in inc[u]:
            v = ends[e][0] if ends[e][1] == u else ends[e][1]
            if d + weights[e] < dist.get(v, 1e9):
                dist[v] = d + weights[e]; heapq.heappush(pq, (dist[v], v))

def to_nodes(path, s):
    out, cur = [s], s
    for e in path:
        cur = ends[e][0] if ends[e][1] == cur else ends[e][1]
        out.append(cur)
    return out

if __name__ == "__main__":
    print("sanity: Machine 6 -> Delivery =",
          dijkstra(machine["machine 6"], machine["Delivery"]), "(expected 100)")
    s, t = machine["machine 5"], machine["machine 3"]
    paths, hits = enumerate_paths(s, t)
    paths.sort(key=lambda p: sum(weights[e] for e in p))
    print(f"simple paths found: {len(paths)}  (solutions with detached cycles rejected: {hits})")
    print("Dijkstra shortest:", dijkstra(s, t))
    for k, p in enumerate(paths[:10], 1):
        print(f"{k:2d}. length {sum(weights[e] for e in p):4d}  ", " -> ".join(to_nodes(p, s)))

sanity: Machine 6 -> Delivery = None (expected 100)
simple paths found: 890  (solutions with detached cycles rejected: 0)
Dijkstra shortest: 114
 1. length  114   (2,2) -> (1,2) -> (1,3) -> (1,4) -> (2,4) -> (2,5) -> (3,5)
 2. length  114   (2,2) -> (1,2) -> (1,3) -> (1,4) -> (1,5) -> (2,5) -> (3,5)
 3. length  114   (2,2) -> (2,3) -> (3,3) -> (3,4) -> (2,4) -> (2,5) -> (3,5)
 4. length  114   (2,2) -> (2,3) -> (3,3) -> (4,3) -> (4,4) -> (4,5) -> (3,5)
 5. length  114   (2,2) -> (2,3) -> (1,3) -> (1,4) -> (1,5) -> (2,5) -> (3,5)
 6. length  114   (2,2) -> (2,3) -> (1,3) -> (1,4) -> (2,4) -> (2,5) -> (3,5)
 7. length  148   (2,2) -> (1,2) -> (1,3) -> (2,3) -> (3,3) -> (4,3) -> (4,4) -> (4,5) -> (3,5)
 8. length  148   (2,2) -> (1,2) -> (1,3) -> (2,3) -> (3,3) -> (3,4) -> (2,4) -> (2,5) -> (3,5)
 9. length  148   (2,2) -> (2,3) -> (3,3) -> (3,4) -> (2,4) -> (1,4) -> (1,5) -> (2,5) -> (3,5)
10. length  152   (2,2) -> (1,2) -> (1,3) -> (1,4) -> (0,4) -> (0,5) -> (1,5) -> (2,5) -> (3,5)
